In [11]:
from langchain_ollama import OllamaLLM
import pandas as pd
import numpy as np
import re
from cleantext import clean
Words2Rmv = ['xxx']

In [2]:
llm = OllamaLLM(model="financial_analyser_llama2:latest")
llm.invoke("Who made a biggest investment in AI in the last 5 years?")

"\nAs a financial assistant, I can provide information on investments in AI made by various companies in the last 5 years. However, it's important to note that the investment landscape is constantly evolving, and new players are entering the field every year. Therefore, the answer to this question may change over time.\n\nBased on my analysis of publicly available data, some of the biggest investments in AI made by well-known companies in the last 5 years include:\n\n1. Google - Google has invested heavily in AI research and development, particularly in the areas of machine learning, natural language processing, and computer vision. The company has made significant investments in startups like DeepMind and NVIDIA, and has also acquired several AI-related companies.\n2. Microsoft - Microsoft has made significant investments in AI through its acquisition of LinkedIn, which provides access to a vast database of professional networking data. The company has also invested in startups like H

In [6]:
def read_data():
    TransDF = pd.read_csv('sample100.csv')
    TransDF['Amount'] = pd.to_numeric(TransDF['Amount'], errors='coerce').fillna(0).astype(int)
    
    return TransDF

AllTransDF = read_data()

#make sure the data has the right data type
print(AllTransDF.dtypes)

Date             str
Amount         int64
Transaction      str
dtype: object


In [9]:
AllTransDF.head()

,Date,Amount,Transaction
0,30/07/2025,-4,BACI PASTA CAFE
1,30/12/2025,-9,KMART
2,07/01/2026,-197,BARWON WATER 1
3,15/01/2025,-62,TONG LI SUPERMARKETS
4,12/11/2025,-634,AGODA.COM ALUNA BEN INTERNET


In [42]:
def transform_data(AllTransDF):
    pd.set_option('display.max_colwidth', None)
    AllTransDF['CleanTransaction'] = AllTransDF['Transaction'].apply(
        lambda x: ' '.join([word for word in x.split() 
                            if word.lower() not in Words2Rmv])
    )
    
    AllTransDF['Transaction'] = AllTransDF['CleanTransaction'].apply(lambda x: clean(
        x,
        clean_all=True,
        extra_spaces=True,
        numbers=True,
        punct=True
        ))
    
    AllTransDF = AllTransDF.drop("CleanTransaction", axis=1)
    AllTransDF['Income/Expense'] = np.where(AllTransDF['Amount']>0, 'Income','Expense')
    AllTransDF['Transaction']=AllTransDF['Transaction'].str.title()
    return (AllTransDF)

TransformedDF = transform_data(AllTransDF)

TransformedDF


,Date,Amount,Transaction,Income/Expense
0,30/07/2025,-4,Baci Pasta Cafe,Expense
1,30/12/2025,-9,Kmart,Expense
2,07/01/2026,-197,Barwon Water,Expense
3,15/01/2025,-62,Tong Li Supermarkets,Expense
4,12/11/2025,-634,Agodacom Aluna Ben Internet,Expense
...,...,...,...,...
95,19/01/2026,-6,Woolworthsgreat Western,Expense
96,25/05/2026,-19,Woolworthswestfield Camp,Expense
97,18/11/2025,-992,Banking Funds Tfer Transfer To Cc,Expense
98,11/11/2024,2167,Transfer From,Income


In [14]:
UniqueTransactions=TransformedDF['Transaction'].unique()
len(UniqueTransactions)
UniqueTransactions

<StringArray>
[                                                'Baci Pasta Cafe',
                                                           'Kmart',
                                                    'Barwon Water',
                                            'Tong Li Supermarkets',
                                     'Agodacom Aluna Ben Internet',
                                           'Aust Hlth Mgmt Grp Pl',
                                      'Mobile Banking Payment  To',
                                                   'Transfer From',
                                                     'Aldi Stores',
                                  'Transfer From Metro Art Galary',
                                                       'Big Apple',
                                         'Woolworthsgreat Western',
                                                           'Coles',
                                          'Payment To Occomptyltd',
                              'Ban

In [17]:
response = llm.invoke('can you add a proper category to the following items. Do not apply reasonong. ' \
'Responde with category. For example: Coles Express Bread - groceries, Big Apple transaction - groceries' + 
', '.join(UniqueTransactions[90:100]))
response = response.split('\n')
response

['',
 'Sure! Please provide the items from bank transactions and I will assist you in categorizing them:',
 '',
 '1. $10.00 from 7-Eleven - Fuel',
 '2. $5.00 from Coles Express - Groceries',
 '3. $15.00 from Big Apple - Entertainment',
 '4. $20.00 from Coffee Shop - Food and Beverage',
 '5. $30.00 from Online Store - Clothing',
 '6. $10.00 from Gas Station - Fuel',
 '7. $15.00 from Restaurant - Entertainment',
 '8. $25.00 from Home Improvement Store - Home and Garden',
 '9. $10.00 from Parking Lot - Transportation',
 '10. $30.00 from Salon - Personal Care']

In [18]:
#Create an ndex list to partiction blocks of transactions, read 10 transaction each time
def hop(start, stop, step):
    for i in range(start, stop, step):
        yield i 
    yield stop
IndexTransList = list(hop(0, len(UniqueTransactions), 10))
IndexTransList

[0, 10, 20, 30, 40, 50, 54]

In [23]:
# validate output
from pydantic import BaseModel, field_validator
from typing import List

class ResponseChecks(BaseModel):
    data: List[str]
    @field_validator('data')
    def check_length(cls, value):
        for item in value:
            if len(item) >0:
                assert "-" in item, f"Item '{item}' does not contain a hyphen"
#if the check will pass
ResponseChecks(data=['Hello World - greeting', 'banana cake - fruit'])
                    
    

ResponseChecks(data=None)

In [20]:
#if the check will fail
ResponseChecks(data=['Hello-World', 'banana fruit'])

ValidationError: 1 validation error for ResponseChecks
data
  Assertion failed, Item 'banana fruit' does not contain a hyphen [type=assertion_error, input_value=['Hello-World', 'banana fruit'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/assertion_error

In [24]:
def categorise_transactions(transactions, llm):
    response = llm.invoke("Can you add an appropriate category to the following expenses. For example: " \
    "Best Baking Syndey Metro - food, oil paint canvas - creative, uberdrive int aiport - transport, utility Sept - utility. " \
    "Categories should be less than 4 characters. " + transactions)
    response = response.split('\n')
    
    #removing the explaination lines at the beginning and end of the response
    TransIndexes = [index for index in range(len(response)) if response[index] == '']
    if len(TransIndexes) == 1:
        response = response[(TransIndexes[0] + 1):]
    else:
        response = response[(TransIndexes[0] + 1) : TransIndexes[1]]

    #validate response for '-' to separate transaction and category
    ResponseChecks(data = response)
    
    # Put in dataframe
    categories_df = pd.DataFrame({'Transaction - Category': response})
    categories_df[['Transaction', 'Category']] = categories_df['Transaction - Category'].str.split(' - ', expand=True)

    return categories_df


In [27]:
#test the categorise_transactions function
trans_cate = categorise_transactions('Best Baking Syndey Metro, oil paint canvas , uberdrive int aiport, utility Sept',llm)
trans_cate

,Transaction - Category,Transaction,Category
0,* Best Baking Sydney Metro - food,* Best Baking Sydney Metro,food
1,* Oil Paint Canvas - creative,* Oil Paint Canvas,creative
2,* Uber Drive Int Airport - transport,* Uber Drive Int Airport,transport
3,* Utility Sept - utility,* Utility Sept,utility


In [28]:
def process_categories(UniqueTransactions, IndexTransList, llm):
  
    #create a new dataframe to store all the categories
    CategoriesDF = pd.DataFrame()
    #initialise max tries if model fails to respond with the correct format, expect to include '-' between transaction and category
    MaxTries = 5

    #loop through the unique transactions and get the categories
    for i in range(0,len(IndexTransList)-1):
        BlockTrans= UniqueTransactions[IndexTransList[i]:IndexTransList[i+1]]
        BlockTrans = ', '.join(BlockTrans)

        for j in range(MaxTries):
            try:
                Categories = categorise_transactions(BlockTrans, llm)
                CategoriesDF = pd.concat([CategoriesDF, Categories], ignore_index=True)                                 

            except:
                if j < MaxTries:
                    continue
                else:
                    raise Exception (f"Failed to categorise transactions after {MaxTries} attempts.")
            break
    return CategoriesDF

CategoriesDF = process_categories(UniqueTransactions, IndexTransList, llm)

In [40]:
CategoriesDF['Transaction'] = CategoriesDF['Transaction'].str.replace(r'^(\d+\.|\*)\s+', '', regex=True).str.strip()
CategoriesDF['Category']=CategoriesDF['Category'].str.title()
CategoriesDF

,Transaction - Category,Transaction,Category
0,* Baci Pasta Cafe - Food,Baci Pasta Cafe,Food
1,* Kmart - Shopping,Kmart,Shopping
2,* Barwon Water - Utility,Barwon Water,Utility
3,* Tong Li Supermarkets - Groceries,Tong Li Supermarkets,Groceries
4,* Agodacom Aluna Ben Internet - Tech/Internet,Agodacom Aluna Ben Internet,Tech/Internet
5,* Aust Hlth Mgmt Grp Pl - Insurance,Aust Hlth Mgmt Grp Pl,Insurance
6,* Mobile Banking Payment To - Financial Services,Mobile Banking Payment To,Financial Services
7,* Transfer From - Financial Services,Transfer From,Financial Services
8,* Aldi Stores - Groceries,Aldi Stores,Groceries
9,* Transfer From Metro Art Gallery - Entertainment,Transfer From Metro Art Gallery,Entertainment


In [33]:
CategorisedTrans = TransformedDF.merge(CategoriesDF[['Transaction','Category']], on='Transaction', how='left')
CategorisedTrans = CategorisedTrans.drop_duplicates(subset=['Date', 'Amount', 'Transaction'], keep='first')
CategorisedTrans


,Date,Amount,Transaction,Income/Expense,Category
0,30/07/2025,-4,Baci Pasta Cafe,Expense,Food
1,30/12/2025,-9,Kmart,Expense,Shopping
2,07/01/2026,-197,Barwon Water,Expense,Utility
3,15/01/2025,-62,Tong Li Supermarkets,Expense,Groceries
4,12/11/2025,-634,Agodacom Aluna Ben Internet,Expense,Tech/Internet
...,...,...,...,...,...
95,19/01/2026,-6,Woolworthsgreat Western,Expense,NaN
96,25/05/2026,-19,Woolworthswestfield Camp,Expense,NaN
97,18/11/2025,-992,Banking Funds Tfer Transfer To Cc,Expense,NaN
98,11/11/2024,2167,Transfer From,Income,Financial Services


In [41]:
CategorisedTransCleaned = CategorisedTrans.dropna()
CategorisedTransCleaned.to_csv('Categorised_cleaned_trans.csv')
CategorisedTransCleaned

,Date,Amount,Transaction,Income/Expense,Category
0,30/07/2025,-4,Baci Pasta Cafe,Expense,Food
1,30/12/2025,-9,Kmart,Expense,Shopping
2,07/01/2026,-197,Barwon Water,Expense,Utility
3,15/01/2025,-62,Tong Li Supermarkets,Expense,Groceries
4,12/11/2025,-634,Agodacom Aluna Ben Internet,Expense,Tech/Internet
...,...,...,...,...,...
89,31/03/2025,2375,Transfer From,Income,Financial Services
91,08/12/2025,-2,Ezymart Central Stn,Expense,shopping
92,15/05/2025,-37,Aldi Stores,Expense,Groceries
93,23/07/2025,-37,Aldi Stores,Expense,Groceries


In [37]:
UniqCategories = CategorisedTransCleaned['Category'].unique()
UniqCategories

<StringArray>
[              'Food',           'Shopping',            'Utility',
          'Groceries',      'Tech/Internet',          'Insurance',
 'Financial Services',             'Retail',            'Grocery',
           'Creative',          'Transport',           'shopping',
          'utilities',          'transport',            'banking',
             'Online',    'Office Supplies',    'Health & Beauty',
            'Banking', 'Telecommunications',           'Clothing']
Length: 21, dtype: str